# Libeniz and Pi

There is a **Leibniz formula** for approximating π

$$
\pi = 4 \sum_{k=0}^{\infty} \frac{(-1)^k}{2k+1}
$$


where the Taylor series for:

$$
arctan(x)
$$
$$
arctan(x) = \sum_{k=0}^{\infty} \frac{(-1)^kx^{2k+1}}{2k+1}
$$

plugin `x = 1`

$$
arctan(1) = \frac{\pi}{4}
$$

so the Leibniz formula, comes directly from a Taylor series expansion.

$$
\pi = 4 (1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + ...)
$$

which is the **Leibniz formula** at the top

In [1]:
def leibniz_pi(terms: int) -> float:
    """
    Approximate pi using the Leibniz series.

    pi = 4 * (1 - 1/3 + 1/5 - 1/7 + ...)

    Args:
        terms: Number of terms to include. Must be >= 1.

    Returns:
        Approximation of pi as a float.
    """
    total = 0.0
    for i in range(terms):
        total += (-1) ** i / (2 * i + 1)
    return 4 * total

running some tests

In [2]:
print(leibniz_pi(10))
print(leibniz_pi(100))
print(leibniz_pi(1000))

3.0418396189294032
3.1315929035585537
3.140592653839794


use `pandas` to make a pretty table

In [3]:
# %pip install pandas
import math
import pandas as pd

rows = []
for n in [10, 100, 1000, 100_000]:
    approx = leibniz_pi(n)
    rows.append({
        "terms": n,
        "leibniz_pi": approx,
        "math.pi": math.pi,
        "error": abs(approx - math.pi)
    })

df = pd.DataFrame(rows)
df


,terms,leibniz_pi,math.pi,error
0,10,3.041840,3.141593,0.099753
1,100,3.131593,3.141593,0.010000
2,1000,3.140593,3.141593,0.001000
3,100000,3.141583,3.141593,0.000010


write a `pytest` test to test it

In [4]:
# %pip install ipytest
import ipytest
import pytest
ipytest.autoconfig()

In [5]:
%%ipytest -vv

def test_libeniz_pi_10_3_0418():
    assert leibniz_pi(10) == pytest.approx(math.pi, abs=0.1)
    assert round(leibniz_pi(10),4) == 3.0418

def test_libeniz_pi_100_3_1316():
    assert leibniz_pi(100) == pytest.approx(math.pi, abs=0.01)
    assert round(leibniz_pi(100),4) == 3.1316

def test_libeniz_pi_1000_3_1406():
    assert leibniz_pi(1000) == pytest.approx(math.pi, abs=0.001)
    assert round(leibniz_pi(1000),4) == 3.1406
    
def test_libeniz_pi_1_000_000_3_1416():
    assert leibniz_pi(1_000_000) == pytest.approx(math.pi, abs=0.00001)
    assert round(leibniz_pi(1_000_000),4) == 3.1416
    


======================================= test session starts ========================================
platform darwin -- Python 3.14.3, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/Caskroom/miniconda/base/envs/ipy_env/bin/python
cachedir: .pytest_cache
rootdir: /Users/michael/Projects/saramic/learning/demo/libeniz-and-pi
plugins: anyio-4.12.1
collecting ... collected 4 items

t_6e3bf982b18e4c95aa9675b1f1cecd92.py::test_libeniz_pi_10_3_0418 PASSED                      [ 25%]
t_6e3bf982b18e4c95aa9675b1f1cecd92.py::test_libeniz_pi_100_3_1316 PASSED                     [ 50%]
t_6e3bf982b18e4c95aa9675b1f1cecd92.py::test_libeniz_pi_1000_3_1406 PASSED                    [ 75%]
t_6e3bf982b18e4c95aa9675b1f1cecd92.py::test_libeniz_pi_1_000_000_3_1416 PASSED               [100%]

======================================== 4 passed in 0.28s =========================================


maybe we could speed up our leibniz function

In [6]:
%%timeit

leibniz_pi(1_000)

69.1 μs ± 775 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
def leibniz_faster_pi(terms: int) -> float:
    total = 0.0
    sign = 1.0
    for i in range(terms):
        total += sign / (2 * i + 1)
        sign = -sign
    return 4 * total

In [8]:
%%timeit

leibniz_faster_pi(1_000)

36.3 μs ± 730 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


the `sign` means we no longer have to do the slightly more expensive `(-1) ** i` and as it is always $1^k$ or $(-1)^k$ we don't even need to do the "to the power of" as it will always be `1` or `-1`

In [12]:
import time

def speed_of(fn, *args, **kwargs):
    start = time.monotonic()
    result = fn(*args, **kwargs)
    end = time.monotonic()
    return result, end - start
    
rows = []
for n in [10, 100, 1000, 100_000]:
    approx, elapsed = speed_of(leibniz_pi, n)
    approx_faster, elapsed_faster = speed_of(leibniz_faster_pi, n)
    rows.append({
        "terms": n,
        "leibniz_pi": approx,
        "leibniz_time_s": elapsed,
        "leibniz_faster_pi": approx_faster,
        "leibniz_faster_time_s": elapsed_faster,
        "faster_by": round(elapsed / elapsed_faster, 2),
    })

df = pd.DataFrame(rows)
df

,terms,leibniz_pi,leibniz_time_s,leibniz_faster_pi,leibniz_faster_time_s,faster_by
0,10,3.041840,0.000006,3.041840,0.000003,1.84
1,100,3.131593,0.000014,3.131593,0.000009,1.50
2,1000,3.140593,0.000174,3.140593,0.000102,1.71
3,100000,3.141583,0.022060,3.141583,0.003595,6.14


but Leibniz is stall slow. It was John Machin (1706) who computed π to 100 digits and is used to get this value with a greater precision. Machin original formual is:

$$
\pi = 16 arctan(\frac{1}{5}) - 4 arctan(\frac{1}{239})
$$

this is because $arctan(\frac{1}{239})$ has a small **x** and that means it converges fater

In [10]:
def arctan_series(x: float, terms: int) -> float:
    total = 0.0
    power = x
    sign = 1.0
    for k in range(terms):
        total += sign * power / (2 * k + 1)
        power *= x * x
        sign = -sign
    return total

def machin_pi(terms: int) -> float:
    return 16.0 * arctan_series(1.0 / 5.0, terms) - 4.0 * arctan_series(1.0 / 239.0, terms)

In [11]:
TARGET_ERROR = 5e-4 # 5e-15 # issues arise going above this? float and rounding

def find_terms_to_accuracy(fn, target_error: float, max_terms: int) -> tuple[int, float]:
    for terms in range(1, max_terms + 1):
        value = fn(terms)
        error = abs(value - math.pi)
        if error < target_error:
            return terms, value
    return terms, value
    # raise RuntimeError(f"Did not reach target error within {max_terms} terms")

rows = []
for pi_func in [machin_pi, leibniz_faster_pi]:
    terms, approx = find_terms_to_accuracy(
        pi_func, TARGET_ERROR, max_terms=1_000
    )
    _, elapsed = speed_of(pi_func, terms)

    rows.append({
        "function": pi_func.__name__,
        "value": approx,
        "time_s": elapsed,
        "terms": terms,
    })

df = pd.DataFrame(rows)
df

,function,value,time_s,terms
0,machin_pi,3.141621,0.000004,3
1,leibniz_faster_pi,3.140593,0.000035,1000


**Machin** can get to with $5*10^-4$ within 3 terms, but **Leibniz**, even with 1,000 terms still does not get to that accuracy